In [ ]:
!nvidia-smi


In [ ]:
!pip install unsloth
!pip install mlflow

In [ ]:
%%writefile finetune_config.yaml
model:
  base_model: "unsloth/Qwen2.5-7B-Instruct"
  max_seq_length: 2048
  load_in_4bit: true

lora:
  r: 16
  lora_alpha: 16
  lora_dropout: 0
  target_modules: ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

data:
  dataset_repo: "saifigohar1122/new-data_set"
  split: "train"

training:
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 4
  num_train_epochs: 2
  learning_rate: 0.0002
  warmup_steps: 5
  lr_scheduler_type: "linear"
  optim: "adamw_8bit"
  weight_decay: 0.01
  logging_steps: 1
  seed: 3407
  output_dir: "outputs"

mlflow:
  enabled: false
  experiment_name: "qwen-qa-finetune"
  tracking_uri: ""
  run_name: "qwen2.5-7b-qa"

output:
  push_to_hub: true
  new_model_repo: "saifigohar1122/qwen2.5-7b-qa-lora"

In [ ]:
!pip install pyyaml

In [ ]:
import yaml
with open("finetune_config.yaml") as f:
    cfg = yaml.safe_load(f)

m   = cfg["model"]
lo  = cfg["lora"]
d   = cfg["data"]
t   = cfg["training"]
mlf = cfg["mlflow"]
out = cfg["output"]
print("Config loaded ->", m["base_model"])

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = m["base_model"],
    max_seq_length = m["max_seq_length"],
    load_in_4bit   = m["load_in_4bit"],
)
print("Model loaded.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lo["r"],
    target_modules = lo["target_modules"],
    lora_alpha = lo["lora_alpha"],
    lora_dropout = lo["lora_dropout"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = t["seed"],
)

**Cell 8 — Dataset load + chat format**

In [ ]:
from datasets import load_dataset

dataset = load_dataset(d["dataset_repo"], split=d["split"])

def to_chat(ex):
    instr = (ex.get("instruction") or "").strip()
    inp   = (ex.get("input") or "").strip()
    user  = instr if not inp else f"{instr}\n\n{inp}"
    msgs = [
        {"role": "user",      "content": user},
        {"role": "assistant", "content": (ex.get("output") or "").strip()},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

dataset = dataset.map(to_chat, remove_columns=dataset.column_names)
print(f"{len(dataset)} records ready.\n")
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = m["max_seq_length"],
        per_device_train_batch_size = t["per_device_train_batch_size"],
        gradient_accumulation_steps = t["gradient_accumulation_steps"],
        num_train_epochs            = t["num_train_epochs"],
        learning_rate               = t["learning_rate"],
        warmup_steps                = t["warmup_steps"],
        lr_scheduler_type           = t["lr_scheduler_type"],
        optim                       = t["optim"],
        weight_decay                = t["weight_decay"],
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps               = t["logging_steps"],
        seed                        = t["seed"],
        output_dir                  = t["output_dir"],
        report_to = ["mlflow"] if mlf["enabled"] else "none",
        run_name  = mlf["run_name"],
    ),
)

trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Explain the DMAS rule in mathematics."}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out_ids = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))